# Confirmatory descriptives (VS study)

Descriptive layer for the confirmatory collection: coverage, per-cell
final-hop summaries, between-chain variance and drift, and cumulative
curve figures. The notebook reads the preprocessing exports produced by
`scripts/preprocessing/write_stats_confirmatory.py` under
`$DATA_DIR/data/confirmatory/stats/<model>/<config>/<task>/<condition>/`
and writes to `$DATA_DIR/pub/confirmatory/descriptives/`.

All summaries use one accepted attempt per seed x chain (the highest
replicate when reruns exist). Cells that are not collected yet are skipped
and listed at the end, never filled in. Variance summaries average over the
three validated axes (valence, factual-to-narrative, abstract-to-concrete).

In [1]:
import json
from datetime import datetime, UTC
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from echochain.diffusion import AXES
from echochain.utils import get_data_dir
from echochain.plot_utils import ci95_seed
from echochain.constants import CONDITION_COLORS as PILOT_COLORS

import pub_utils.figures as F
from pub_utils.tables import TableSpec, write_table

from transformers import logging

logging.set_verbosity_error()

In [2]:
DATA_DIR = Path(get_data_dir())
STATS_ROOT = DATA_DIR / "data" / "confirmatory" / "stats"
PUB_DIR = DATA_DIR / "pub" / "confirmatory" / "descriptives"
PUB_DIR.mkdir(parents=True, exist_ok=True)
SEEDS_FILE = DATA_DIR / "data" / "confirmatory" / "corpus" / "sampler" / "vs_confirmatory_seeds_final.jsonl"

TASKS = ["paraphrase", "improve", "take_inspiration", "expand+compress"]
TASK_LABELS = {
    "paraphrase": "Paraphrase",
    "improve": "Improve",
    "take_inspiration": "Take inspiration",
    "expand+compress": "Expand + compress",
}
CONDITIONS = ["direct", "uniform_list", "vs_weighted"]
CONDITION_LABELS = {
    "direct": "Direct",
    "uniform_list": "Multi-k uniform",
    "vs_weighted": "VS",
}
COND_COLORS = {
    "direct": PILOT_COLORS["direct"],
    "uniform_list": PILOT_COLORS.get("uniform_list", PILOT_COLORS["multi_k"]),
    "vs_weighted": PILOT_COLORS["vs_weighted"],
}
VALIDATED_AXES = [
    "valence_neg_to_pos",
    "factual_to_narrative",
    "abstract_to_concrete",
]
AXIS_LABELS = {
    "valence_neg_to_pos": "Valence",
    "tone_neutral_to_intense": "Tone intensity",
    "moderate_to_extreme": "Extremity",
    "factual_to_narrative": "Factual vs narrative",
    "abstract_to_concrete": "Abstract vs concrete",
}
HOP = 200
KEY_METRICS = [
    "delta_len",
    "ratio_len",
    "jaccard",
    "entity_f1",
    "triple_f1",
    "cosine",
    "entail_ab",
    "entail_ba",
    "entail_consistent",
    "k_unique",
]
TITLES = {
    "delta_len": r"Delta tokens",
    "ratio_len": "Expansion ratio",
    "jaccard": "Lexical Jaccard",
    "entity_f1": "Entity F1",
    "triple_f1": "Triple F1",
    "cosine": "Embedding cosine",
    "entail_ab": "Seed-to-hop entailment",
    "entail_ba": "Hop-to-seed entailment",
    "entail_consistent": "Mutual entailment",
    "k_unique": "Distinct texts per chain",
}

In [3]:
def accepted(df):
    """One accepted attempt per (msg_id, chain): the highest replicate.

    Reruns write `_repN` logs; the frozen rerun policy replaces a failed or
    flagged pair with the next replicate. `rep` is kept so callers can report
    how many pairs were replaced.
    """
    d = df.copy()
    d["rep"] = d["rep"].fillna(-1).astype(int) if "rep" in d else -1
    return d.sort_values(["msg_id", "chain", "rep"]).drop_duplicates(
        ["msg_id", "chain"], keep="last"
    )


def cell_key(c):
    return f"{c['model']}|{c['config']}|{c['task']}|{c['condition']}"


def safe(name):
    return name.replace("/", "-").replace(".", "p").replace("+", "plus")


def load_last_hop(c):
    return accepted(pd.read_csv(c["dir"] / "metrics_last_hop_cumulative.csv"))


def load_axis(c):
    return accepted(pd.read_csv(c["dir"] / "axis_cumulative_long.csv"))


def load_metrics(c, metrics):
    cols = ["metric", "msg_id", "chain", "rep", "hop", "value"]
    df = pd.read_csv(c["dir"] / "metrics_cumulative_long.csv", usecols=cols)
    return accepted(df[df["metric"].isin(metrics)])

## 1. Coverage

In [4]:
def discover_cells():
    cells = []
    for csv_path in sorted(STATS_ROOT.glob("**/metrics_cumulative_long.csv")):
        p = csv_path.parent
        rel = p.relative_to(STATS_ROOT)
        parts = rel.parts
        if len(parts) < 4:
            continue
        model = "/".join(parts[:-3])
        config, task, condition = parts[-3:]
        if task not in TASKS or condition not in CONDITIONS:
            continue
        cells.append(
            {
                "model": model,
                "config": config,
                "task": task,
                "condition": condition,
                "dir": p,
            }
        )
    return cells


cells = discover_cells()
print(f"{len(cells)} cell(s) with exports under {STATS_ROOT}")
if not cells:
    print(
        "WARNING: no preprocessing exports found. Run "
        "scripts/preprocessing/write_stats_confirmatory.py on the GPU "
        "workstation first; the tables and figures below will be empty."
    )
for model in sorted({c["model"] for c in cells}):
    for task in TASKS:
        have = [
            cond
            for cond in CONDITIONS
            if any(
                c["model"] == model and c["task"] == task and c["condition"] == cond
                for c in cells
            )
        ]
        pending = [cond for cond in CONDITIONS if cond not in have]
        print(f"  {model} / {TASK_LABELS[task]}: collected={have} pending={pending}")

0 cell(s) with exports under /home/user/Dropbox/data_export/echochains/data/confirmatory/stats


In [ ]:
cov_rows = []
for c in cells:
    lh = load_last_hop(c)
    cov_file = c["dir"] / "coverage.json"
    cov = json.loads(cov_file.read_text(encoding="utf-8")) if cov_file.exists() else None
    cov_rows.append(
        {
            "Model": c["model"],
            "Task": TASK_LABELS[c["task"]],
            "Condition": CONDITION_LABELS[c["condition"]],
            "Seeds": lh["msg_id"].nunique(),
            "Chains": lh[["msg_id", "chain"]].drop_duplicates().shape[0],
            "Expected chains": (
                cov["expected"]["chains"]
                if cov and cov["expected"]["chains"]
                else np.nan
            ),
            "Partial logs": cov["found"]["partial_logs"] if cov else np.nan,
            "Failed records": cov["found"]["failed_records"] if cov else np.nan,
        }
    )
coverage = pd.DataFrame(cov_rows)
coverage

In [ ]:
if not coverage.empty:
    disp = coverage.copy()
    for col in ["Expected chains", "Partial logs", "Failed records"]:
        disp[col] = disp[col].map(lambda v: "" if pd.isna(v) else int(v))
    spec = TableSpec(
        caption="Confirmatory coverage per model, task, and condition at export time. Chains counts accepted seed $\\times$ chain pairs; expected chains is 90 seeds $\\times$ 5 chains. Partial logs and failed records are recorded by the chain generator and are excluded from the analysis.",
        label="tab:coverage_confirmatory",
        wrap="threeparttable",
        width="2col",
        use_tabularx=False,
        placement="htbp",
        use_siunitx=False,
        use_index=False,
        fontsize_pt=(9, 11),
    )
    write_table(disp, PUB_DIR / "tab_coverage_confirmatory.tex", spec)
    print("wrote", PUB_DIR / "tab_coverage_confirmatory.tex")

## 2. Final-hop summaries

One table per cell with seed-clustered 95% intervals (chains within a seed
are not independent observations). The same values go into
`metrics_pub.json`.

In [ ]:
metrics_pub = {
    "meta": {
        "generated_at": datetime.now(UTC).isoformat(),
        "tasks": TASKS,
        "conditions": CONDITIONS,
        "validated_axes": VALIDATED_AXES,
        "hop": HOP,
    },
    "coverage": cov_rows,
    "cells": {},
}

for c in cells:
    lh = load_last_hop(c)
    rows = []
    for metric, g in lh.groupby("metric"):
        if metric not in KEY_METRICS:
            continue
        mean, h95, n_seeds = ci95_seed(g["value"].values, g["msg_id"].values)
        rows.append(
            dict(
                metric=metric,
                mean=mean,
                ci_lo=mean - h95,
                ci_hi=mean + h95,
                N=len(g),
                n_seeds=n_seeds,
            )
        )
    tbl = (
        pd.DataFrame(rows)
        .set_index("metric")
        .reindex(KEY_METRICS)
        .reset_index()
        .dropna(subset=["mean"])
    )
    disp = tbl[["metric", "mean", "ci_lo", "ci_hi", "N"]].copy()
    disp.columns = ["Metric", "Mean", "CI low", "CI high", "N"]
    disp["Metric"] = disp["Metric"].map(TITLES)
    tag = f"{safe(c['model'])}_{safe(c['config'])}_{safe(c['task'])}_{safe(c['condition'])}"
    spec = TableSpec(
        caption=f"{c['model']} ({c['config'].replace('_', ' ')}) — {TASK_LABELS[c['task']]} — {CONDITION_LABELS[c['condition']]} — final-hop values",
        label=f"tab:lasthop_confirmatory_{tag}",
        wrap="threeparttable",
        width="2col",
        use_tabularx=False,
        placement="htbp",
        use_siunitx=True,
        float_fmt="{:.3f}",
        use_index=False,
        fontsize_pt=(10, 12),
        numeric_as_r={"N"},
    )
    out = PUB_DIR / f"tab_lasthop_confirmatory_{tag}.tex"
    write_table(disp, out, spec)
    print("wrote", out)
    metrics_pub["cells"][cell_key(c)] = {
        "sample": {
            "n_chains": int(lh[["msg_id", "chain"]].drop_duplicates().shape[0]),
            "n_seeds": int(lh["msg_id"].nunique()),
            "n_chains_mean": float(
                lh.groupby("msg_id")["chain"].nunique().mean()
            ),
        },
        "last_hop": {
            r["metric"]: dict(
                mean=float(r["mean"]),
                ci_lo=float(r["ci_lo"]),
                ci_hi=float(r["ci_hi"]),
                N=int(r["N"]),
                n_seeds=int(r["n_seeds"]),
            )
            for r in tbl.to_dict("records")
        },
    }

## 3. Cumulative curves per task

One figure per task and measurement layer, rows for the collected
conditions. Conditions that are not collected yet are simply absent.

In [ ]:
LAYERS = {
    "surface": (["delta_len", "ratio_len", "jaccard"], [False, False, True]),
    "factual": (["entity_f1", "triple_f1"], [True, True]),
    "semantic": (
        ["cosine", "entail_ab", "entail_ba", "entail_consistent"],
        [True, True, True, True],
    ),
}

F.setup_style(
    profile="nature", use_tex=False, base_font=7, minor_ticks=False, title_font_delta=0
)


def plot_row(axs, by_condition, conditions, measure, bounded):
    for ax, condition in zip(axs, conditions):
        ax.tick_params(top=False, right=False)
        df = by_condition.get(condition)
        if df is None or df.empty:
            ax.set_axis_off()
            continue
        d = df[df["metric"] == measure]
        xs, mu, lo, hi = [], [], [], []
        for hop, g in d.groupby("hop"):
            m, h95, _ = ci95_seed(g["value"].values, g["msg_id"].values)
            xs.append(hop)
            mu.append(m)
            lo.append(m - h95)
            hi.append(m + h95)
        ax.plot(xs, mu, color=COND_COLORS[condition])
        ax.fill_between(xs, lo, hi, color=COND_COLORS[condition], alpha=0.2)
        if bounded:
            ax.set_ylim(0, 1)
        if measure == "delta_len":
            ax.axhline(0, color="#888", lw=0.6, ls="--", zorder=0)


for model in sorted({c["model"] for c in cells}):
    for task in TASKS:
        task_cells = [c for c in cells if c["model"] == model and c["task"] == task]
        if not task_cells:
            continue
        conditions = [
            cond
            for cond in CONDITIONS
            if any(c["condition"] == cond for c in task_cells)
        ]
        for layer, (measures, bounded) in LAYERS.items():
            by_condition = {
                cond: load_metrics(
                    [c for c in task_cells if c["condition"] == cond][0], measures
                )
                for cond in conditions
            }
            fig, axs = F.make_grid(
                width_mm=193,
                panels=(len(measures), len(conditions)),
                panel_aspect=0.55,
                margins=(0.10, 0.06, 0.985, 0.94),
                gutter=(0.05, 0.10),
                constrained=False,
                flatten=False,
                sharey="row",
                sharex=True,
            )
            grid = np.atleast_2d(axs)
            for r, (measure, bound) in enumerate(zip(measures, bounded)):
                row = grid[r]
                plot_row(row, by_condition, conditions, measure, bound)
                row[0].set_ylabel(TITLES.get(measure, measure), rotation=90)
                if r == len(measures) - 1:
                    for ax in row:
                        ax.set_xlabel("Hop")
            for ax, cond in zip(grid[0], conditions):
                ax.set_title(CONDITION_LABELS[cond])
            fname = f"fig_confirmatory_cum_{safe(model)}_{layer}"
            F.save(fig, PUB_DIR / fname, formats=("pdf", "png"), dpi_png=600, tight=True)
            plt.show()
            plt.close()
            print("wrote", PUB_DIR / f"{fname}.pdf")

## 4. Between-chain variance and drift

Variance at hop 200 uses seeds with all five accepted chains and averages
over the three validated axes. Drift slopes are ordinary least squares fits
of the axis score on hop index over the seed at hop 0 and the 200
transmitted hops.

In [ ]:
def cell_sigma_mu(c):
    ax = load_axis(c)
    if ax.empty:
        return None
    last = ax[ax["hop"] == HOP]
    counts = last.groupby("msg_id")["chain"].nunique()
    complete = counts[counts == 5].index
    if len(complete) == 0:
        return None
    var = last[last["msg_id"].isin(complete)].groupby("msg_id")[AXES].var(ddof=1)
    sigma_per_axis = var.mean()

    def slopes(grp):
        h = grp["hop"].to_numpy(dtype=float)
        return pd.Series(
            {
                axis: float(np.polyfit(h, grp[axis].to_numpy(dtype=float), 1)[0])
                for axis in AXES
            }
        )

    sl = ax.groupby(["msg_id", "chain"])[["hop", *AXES]].apply(slopes)
    mu_per_axis = sl.mean()
    return {
        "n_seeds_complete": int(len(complete)),
        "sigma": {
            "mean_validated": float(sigma_per_axis[VALIDATED_AXES].mean()),
            "per_axis": {axis: float(sigma_per_axis[axis]) for axis in AXES},
        },
        "mu": {
            "per_axis": {axis: float(mu_per_axis[axis]) for axis in AXES},
            "mean_abs_validated": float(mu_per_axis[VALIDATED_AXES].abs().mean()),
        },
    }


sigma_rows = []
for c in cells:
    res = cell_sigma_mu(c)
    if res is None:
        print(f"  {cell_key(c)}: no seed with five accepted chains; sigma/mu skipped")
        continue
    entry = metrics_pub["cells"].setdefault(cell_key(c), {})
    entry["sigma"] = {**res["sigma"], "n_seeds_complete": res["n_seeds_complete"]}
    entry["mu"] = res["mu"]
    sigma_rows.append(
        {
            "Model": c["model"],
            "Task": TASK_LABELS[c["task"]],
            "Condition": CONDITION_LABELS[c["condition"]],
            **{AXIS_LABELS[axis]: res["sigma"]["per_axis"][axis] for axis in AXES},
            "Mean (validated)": res["sigma"]["mean_validated"],
            "Seeds": res["n_seeds_complete"],
        }
    )
sigma_df = pd.DataFrame(sigma_rows)
sigma_df

In [ ]:
if not sigma_df.empty:
    spec = TableSpec(
        caption="Between-chain variance $\\Sigma$ at hop 200 per model, task, and condition. The mean column averages the three validated axes; seeds enter only when all five accepted chains are present. Per-axis columns report all five axes.",
        label="tab:sigma_confirmatory",
        wrap="threeparttable",
        width="2col",
        use_tabularx=False,
        placement="htbp",
        use_siunitx=True,
        fontsize_pt=(8, 10),
        float_fmt="{:.4f}",
        use_index=False,
        numeric_as_r={"Seeds"},
    )
    write_table(sigma_df, PUB_DIR / "tab_sigma_confirmatory.tex", spec)
    print("wrote", PUB_DIR / "tab_sigma_confirmatory.tex")

In [ ]:
if cells:
    out_json = PUB_DIR / "metrics_pub.json"
    out_json.write_text(
        json.dumps(metrics_pub, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print("wrote", out_json)
else:
    print("no exports; metrics_pub.json not written")

print(f"\n{len(cells)} cell(s):")
for c in cells:
    entry = metrics_pub["cells"].get(cell_key(c), {})
    sample = entry.get("sample", {})
    print(
        f"  {cell_key(c)}: {sample.get('n_chains', '?')} chains, "
        f"{sample.get('n_seeds', '?')} seeds"
    )

missing = [
    f"{model}/{TASK_LABELS[task]}/{CONDITION_LABELS[cond]}"
    for model in sorted({c["model"] for c in cells})
    for task in TASKS
    for cond in CONDITIONS
    if not any(
        c["model"] == model and c["task"] == task and c["condition"] == cond
        for c in cells
    )
]
if missing:
    print("\npending cells (not collected):")
    for m in missing:
        print("  " + m)